In [1]:
import pandas as pd
import yfinance as yf
import numpy as np
import ta
import plotly.graph_objects as go

# Download historical data for IJR for the last 2 months with 30-minute intervals
symbol = 'IJR'
data = yf.download(tickers=symbol, period='5d', interval='30m')

C:\venv\Investment\investment-311v1\Lib\site-packages\yfinance\scrapers\history.py:239: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  quotes2 = quotes.resample('30T')
[*********************100%%**********************]  1 of 1 completed


In [2]:
data.tail()

,Open,High,Low,Close,Adj Close,Volume
Datetime,,,,,,
2024-07-29 13:30:00,116.650002,116.660004,116.480003,116.629997,116.629997,125900
2024-07-29 14:00:00,116.580002,116.800003,116.500099,116.695000,116.695000,270125
2024-07-29 14:30:00,116.714996,116.769997,116.504997,116.690002,116.690002,175545
2024-07-29 15:00:00,116.680000,116.959999,116.629997,116.886299,116.886299,217921
2024-07-29 15:30:00,116.889999,117.260002,116.855797,116.959999,116.959999,377171


In [3]:
macd = ta.trend.MACD(data['Adj Close'])

In [4]:
data['MACD'] = macd.macd()
data['signal'] = macd.macd_signal()
data.tail()

,Open,High,Low,Close,Adj Close,Volume,MACD,signal
Datetime,,,,,,,,
2024-07-29 13:30:00,116.650002,116.660004,116.480003,116.629997,116.629997,125900,0.070349,0.215152
2024-07-29 14:00:00,116.580002,116.800003,116.500099,116.695000,116.695000,270125,0.038057,0.179733
2024-07-29 14:30:00,116.714996,116.769997,116.504997,116.690002,116.690002,175545,0.011924,0.146171
2024-07-29 15:00:00,116.680000,116.959999,116.629997,116.886299,116.886299,217921,0.006973,0.118331
2024-07-29 15:30:00,116.889999,117.260002,116.855797,116.959999,116.959999,377171,0.008893,0.096444


In [5]:
# Initialize columns
data['Position'] = 0
data['Buy_Signal'] = 0
data['Sell_Signal'] = 0
data['Stop_Loss_Signal'] = 0


In [6]:
# Define the look-back window
buy_look_back_window = 2  # Set this to any desired value
stop_loss_threshold = 0.002
# Generate buy, sell, and stop-loss signals
for i in range(buy_look_back_window, len(data)):
    # Check if all MACD values in the look-back window are less than or equal to Signal_Line values
    macd_crossed_above = all(data['MACD'].iloc[i-j] <= data['signal'].iloc[i-j] for j in range(1, buy_look_back_window + 1))
    print(data.iloc[1], macd_crossed_above)
    

Open                   115.300003
High                   116.410004
Low                    115.300003
Close                  116.272102
Adj Close              116.272102
Volume              202167.000000
MACD                          NaN
signal                        NaN
Position                 0.000000
Buy_Signal               0.000000
Sell_Signal              0.000000
Stop_Loss_Signal         0.000000
Name: 2024-07-23 10:00:00, dtype: float64 False
Open                   115.300003
High                   116.410004
Low                    115.300003
Close                  116.272102
Adj Close              116.272102
Volume              202167.000000
MACD                          NaN
signal                        NaN
Position                 0.000000
Buy_Signal               0.000000
Sell_Signal              0.000000
Stop_Loss_Signal         0.000000
Name: 2024-07-23 10:00:00, dtype: float64 False
Open                   115.300003
High                   116.410004
Low                 

In [2]:
# Calculate MACD and generate buy and sell signals
def calculate_macd(df, fast_period=12, slow_period=26, signal_period=9):
    df['EMA_12'] = df['Close'].ewm(span=fast_period, adjust=False).mean()
    df['EMA_26'] = df['Close'].ewm(span=slow_period, adjust=False).mean()
    df['MACD'] = df['EMA_12'] - df['EMA_26']
    df['Signal_Line'] = df['MACD'].ewm(span=signal_period, adjust=False).mean()
    df['Histogram'] = df['MACD'] - df['Signal_Line']
    return df

In [3]:
data = data[['Close']].copy()
data = calculate_macd(data)


In [4]:
# Initialize columns
data['Position'] = 0
data['Buy_Signal'] = 0
data['Sell_Signal'] = 0
data['Stop_Loss_Signal'] = 0


In [11]:
import ta

In [9]:
c

Close               116.272102
EMA_12              115.491861
EMA_26              115.418302
MACD                  0.073558
Signal_Line           0.014712
Histogram             0.058847
Position              0.000000
Buy_Signal            0.000000
Sell_Signal           0.000000
Stop_Loss_Signal      0.000000
Name: 2024-07-23 10:00:00, dtype: float64 False
Close               116.272102
EMA_12              115.491861
EMA_26              115.418302
MACD                  0.073558
Signal_Line           0.014712
Histogram             0.058847
Position              0.000000
Buy_Signal            0.000000
Sell_Signal           0.000000
Stop_Loss_Signal      0.000000
Name: 2024-07-23 10:00:00, dtype: float64 False
Close               116.272102
EMA_12              115.491861
EMA_26              115.418302
MACD                  0.073558
Signal_Line           0.014712
Histogram             0.058847
Position              0.000000
Buy_Signal            0.000000
Sell_Signal           0.000000
Stop_

In [6]:
data[data['Buy_Signal'] == 1]

,Close,EMA_12,EMA_26,MACD,Signal_Line,Histogram,Position,Buy_Signal,Sell_Signal,Stop_Loss_Signal
Datetime,,,,,,,,,,
2024-07-25 11:00:00,116.150002,115.611025,115.681545,-0.070521,-0.107182,0.036661,1,1,0,0


In [7]:
data[data['Sell_Signal'] == 1]

,Close,EMA_12,EMA_26,MACD,Signal_Line,Histogram,Position,Buy_Signal,Sell_Signal,Stop_Loss_Signal
Datetime,,,,,,,,,,
2024-07-25 14:30:00,115.900002,116.232821,116.048226,0.184594,0.126817,0.057777,0,0,1,1


In [ ]:
if data['MACD'].iloc[i] > data['Signal_Line'].iloc[i] and macd_crossed_above and data['Position'].iloc[i-1] == 0:
        data.at[data.index[i], 'Buy_Signal'] = 1
        data.at[data.index[i], 'Position'] = 1
        buy_price = data['Close'].iloc[i]
    elif data['Position'].iloc[i-1] == 1 and (data['MACD'].iloc[i] < data['Signal_Line'].iloc[i] or data['Close'].iloc[i] <= buy_price * (1 - stop_loss_threshold)):
        data.at[data.index[i], 'Sell_Signal'] = 1
        if data['Close'].iloc[i] <= buy_price * (1 - stop_loss_threshold):
            data.at[data.index[i], 'Stop_Loss_Signal'] = 1
        data.at[data.index[i], 'Position'] = 0
    else:
        data.at[data.index[i], 'Position'] = data['Position'].iloc[i-1]